In [1]:
# 常规训练不需要 pip install；Kaggle GPU 镜像通常已含 torch/torchvision/numpy/pandas/Pillow。
import importlib.util
import sys
from pathlib import Path

missing = [name for name in ["torch", "torchvision", "numpy", "pandas", "PIL"]
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"缺少依赖 {missing}；请选择 Kaggle 的 PyTorch GPU 环境，或安装 requirements.txt。")

# None：在 /kaggle/input 中自动查找两个 CSV 同在的目录。
# 若添加了多个版本的数据集，请指定，例如 /kaggle/input/your-dataset/detection。
DATA_ROOT = None
OUTPUT_DIR = "/kaggle/working/fovea_output"

IMAGE_SIZE = 512       # 显存不足先把 BATCH_SIZE 改为 4；尺寸必须是 32 的倍数
BATCH_SIZE = 8
EPOCHS = 80
PATIENCE = 20
N_SPLITS = 5
FOLDS = [0, 1, 2, 3, 4]  # 想先试跑一折可改为 [0]；此时验证分数只覆盖该折
NUM_WORKERS = 2
SEED = 42

PRETRAINED = True
# Internet 关闭时：将 torchvision 官方 resnet18-f37072fd.pth 作为 Kaggle 数据集添加，
# 再填写其完整路径。此参数接收标准 ResNet18 state_dict，不是 fold_*_best.pth。
PRETRAINED_PATH = None
# 例如："/kaggle/input/resnet18-weights/resnet18-f37072fd.pth"
# 不会在权重下载失败后静默切换随机初始化；PRETRAINED=False 才会从零训练。

TTA = True
SMOKE_TEST = False     # True：128尺寸、4张训练图、1轮，单独输出不可提交的检查CSV
PREDICT_ONLY = False   # True：从 OUTPUT_DIR 的原始分折表和已有 fold_*_best.pth 重做预测

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")


In [2]:
%%writefile /kaggle/working/fovea_data.py
"""Data validation, geometry, grouped folds and exact submission formatting.

This module deliberately has no PyTorch dependency, so it can be checked on CPU.
"""
from __future__ import annotations

import hashlib
import json
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageEnhance, ImageOps

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def image_id(value):
    """0001.jpg, 1 and 1.0 all map to the same numeric identifier."""
    value = str(value).strip().replace("\\", "/").split("/")[-1]
    value = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff)$", "", value, flags=re.I)
    if re.fullmatch(r"\d+(?:\.0+)?", value):
        return str(int(float(value)))
    raise ValueError(f"Unexpected image identifier: {value!r}")


def discover_root(requested=None):
    if requested:
        root = Path(requested)
        if not (root / "sample_submission.csv").is_file():
            raise FileNotFoundError(f"DATA_ROOT must contain sample_submission.csv: {root}")
        return root
    candidates = []
    for parent in [Path("/kaggle/input"), Path.cwd()]:
        if parent.exists():
            candidates.extend(p.parent for p in parent.rglob("sample_submission.csv")
                              if (p.parent / "fovea_localization_train_GT.csv").is_file()
                              and (p.parent / "train").is_dir()
                              and (p.parent / "test").is_dir())
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) != 1:
        raise RuntimeError(f"Found {len(candidates)} datasets: {candidates}. Set DATA_ROOT explicitly.")
    return candidates[0]


def image_index(folder):
    result = {}
    for path in sorted(Path(folder).iterdir()):
        if path.suffix.lower() not in IMAGE_EXTENSIONS or not path.is_file():
            continue
        key = image_id(path.name)
        if key in result:
            raise ValueError(f"Duplicate image ID {key} in {folder}")
        result[key] = path
    if not result:
        raise ValueError(f"No images in {folder}")
    return result


def parse_submission_key(key):
    match = re.fullmatch(r"(.+)_Fovea_([XY])", str(key))
    if not match:
        raise ValueError(f"Unexpected sample ImageID: {key!r}")
    return image_id(match[1]), 0 if match[2] == "X" else 1


def inspect_dataset(root, output_dir=None):
    root = Path(root)
    frame = pd.read_csv(root / "fovea_localization_train_GT.csv")
    sample = pd.read_csv(root / "sample_submission.csv", dtype={"ImageID": str})
    if list(frame.columns) != ["data", "Fovea_X", "Fovea_Y"]:
        raise ValueError(f"Unexpected label columns: {list(frame.columns)}")
    if list(sample.columns) != ["ImageID", "value"] or sample.ImageID.duplicated().any():
        raise ValueError("Sample must have unique ImageID rows and columns ImageID,value")
    if frame.isna().any().any():
        raise ValueError("Missing training labels")
    frame["id"] = frame["data"].map(image_id)
    if frame.id.duplicated().any():
        raise ValueError("Duplicate label IDs")
    train, test = image_index(root / "train"), image_index(root / "test")
    if set(frame.id) != set(train):
        raise ValueError("Training image IDs and label IDs do not match")
    if set(train) & set(test):
        raise ValueError("Train/test image IDs overlap")
    requested = [parse_submission_key(k) for k in sample.ImageID]
    if set(requested) != {(k, a) for k in test for a in (0, 1)} or len(requested) != 2 * len(test):
        raise ValueError("Sample must contain exactly one X and one Y row per test image")
    sizes, hashes, duplicates = {}, {}, {}
    for split, index in [("train", train), ("test", test)]:
        for key, path in index.items():
            with Image.open(path) as im:
                im.load()  # Also checks whether the image can actually be decoded.
                sizes[key] = im.size
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            hashes[key] = digest
            duplicates.setdefault(digest, []).append(f"{split}/{path.name}")
    frame["path"] = frame.id.map(lambda k: str(train[k]))
    frame["width"] = frame.id.map(lambda k: sizes[k][0])
    frame["height"] = frame.id.map(lambda k: sizes[k][1])
    frame["group"] = frame.id.map(hashes)
    coords = frame[["Fovea_X", "Fovea_Y"]].to_numpy(dtype=float)
    wh = frame[["width", "height"]].to_numpy(dtype=float)
    if not np.isfinite(coords).all() or not ((coords >= 0) & (coords <= wh - 1)).all():
        raise ValueError("Nonfinite or out-of-image training coordinates")
    offsets = []
    annotation_dir = root / "train_location"
    if annotation_dir.is_dir():
        xml_index = {image_id(p.stem): p for p in annotation_dir.glob("*.xml")}
        for row in frame.itertuples():
            if row.id not in xml_index:
                raise ValueError(f"Missing XML for image {row.id}")
            annotation = ET.parse(xml_index[row.id]).getroot()
            if (int(annotation.findtext("size/width")), int(annotation.findtext("size/height"))) != (row.width, row.height):
                raise ValueError(f"XML size disagrees with image {row.id}")
            boxes = [o.find("bndbox") for o in annotation.findall("object")
                     if o.findtext("name", "").lower() == "fovea"]
            if len(boxes) != 1 or boxes[0] is None:
                raise ValueError(f"Expected one Fovea box for image {row.id}")
            box = [float(boxes[0].findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")]
            if not (box[0] <= row.Fovea_X <= box[2] and box[1] <= row.Fovea_Y <= box[3]):
                raise ValueError(f"CSV coordinate outside XML box for image {row.id}")
            offsets.append(float(np.linalg.norm(np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2])
                                                - [row.Fovea_X, row.Fovea_Y])))
    audit = {
        "root": str(root), "train_images": len(train), "test_images": len(test),
        "sample_rows": len(sample), "sample_columns": list(sample.columns),
        "image_sizes": {f"{w}x{h}": list(sizes.values()).count((w, h)) for w, h in sorted(set(sizes.values()))},
        "identical_files": [v for v in duplicates.values() if len(v) > 1],
        "xml_checked": len(offsets), "label_source": "fovea_localization_train_GT.csv (exact points)",
        "xml_center_offset_median_px": float(np.median(offsets)) if offsets else None,
        "xml_center_offset_max_px": max(offsets) if offsets else None,
    }
    if output_dir is not None:
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        (out / "dataset_audit.json").write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8")
    return frame, sample, test, sizes, audit


def assign_folds(frame, n_splits=5, seed=42):
    """Balance image-size/coordinate strata while keeping identical files together."""
    rng = np.random.default_rng(seed)
    groups = []
    strata = (frame.width.astype(str) + "_" + ((frame.Fovea_X / (frame.width - 1)) < 0.45).astype(str))
    stratum_names = sorted(set(strata))
    for _, rows in frame.groupby("group", sort=True):
        indices = rows.index.to_numpy()
        histogram = np.array([(strata.loc[indices] == name).sum() for name in stratum_names])
        groups.append((indices, histogram, rng.random()))
    if n_splits < 2 or len(groups) < n_splits:
        raise ValueError("Need at least two folds and at least one independent image group per fold")
    # Larger groups first; rare categories are assigned before abundant categories.
    totals = np.array([(strata == name).sum() for name in stratum_names])
    groups.sort(key=lambda g: (-len(g[0]), float((g[1] * totals).sum()), g[2]))
    counts = np.zeros((n_splits, len(stratum_names)), dtype=int)
    sizes = np.zeros(n_splits, dtype=int)
    assignments = np.full(len(frame), -1, dtype=int)
    for indices, histogram, _ in groups:
        tie_break = rng.random(n_splits)
        fold = min(range(n_splits), key=lambda f: (float((counts[f] * histogram).sum()), sizes[f], tie_break[f]))
        assignments[indices] = fold
        counts[fold] += histogram
        sizes[fold] += len(indices)
    if (sizes == 0).any() or (assignments < 0).any():
        raise RuntimeError("Invalid fold assignment")
    if pd.DataFrame({"group": frame.group, "fold": assignments}).groupby("group").fold.nunique().max() != 1:
        raise RuntimeError("Identical images leaked across folds")
    return assignments


def load_resized(path, size):
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        # Antialias the large downsampling first, then correct the small difference
        # between resize's half-pixel map and x/(width-1) endpoint coordinates.
        im = im.resize((size, size), Image.Resampling.BILINEAR)
        sx, sy = (w - 1) * size / ((size - 1) * w), (h - 1) * size / ((size - 1) * h)
        bx, by = 0.5 * size / w - 0.5, 0.5 * size / h - 0.5
        im = im.transform((size, size), Image.Transform.AFFINE,
                          (sx, 0, bx + 0.5 * (1 - sx), 0, sy, by + 0.5 * (1 - sy)),
                          Image.Resampling.BILINEAR)
        return np.asarray(im).copy()


def transform_sample(array, xy, rng):
    """Apply the SAME forward affine map to the image and its normalized point.

    A pixel-center convention is used end to end: normalized x = x/(width-1).
    PIL takes the inverse map for resampling; point labels use the forward map.
    """
    height, width = array.shape[:2]
    center = np.array([(width - 1) / 2, (height - 1) / 2], dtype=float)
    point = np.asarray(xy, dtype=float) * [width - 1, height - 1]
    matrix, transformed = np.eye(3), point
    for _ in range(20):
        angle = np.deg2rad(rng.uniform(-18, 18))
        scale = rng.uniform(0.9, 1.1)
        flip = -1 if rng.random() < 0.5 else 1
        rotation = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
        linear = scale * rotation @ np.diag([flip, 1])
        offset = center + rng.uniform(-0.04, 0.04, size=2) * [width, height] - linear @ center
        proposed = linear @ point + offset
        if ((proposed >= 0) & (proposed <= [width - 1, height - 1])).all():
            matrix[:2, :2], matrix[:2, 2], transformed = linear, offset, proposed
            break
    # PIL's affine coordinates refer to pixel corners; conjugate by a half-pixel
    # translation so the label and resampling both use pixel centers.
    half = np.eye(3)
    half[:2, 2] = 0.5
    pil_inverse = np.linalg.inv(half @ matrix @ np.linalg.inv(half))
    im = Image.fromarray(array).transform((width, height), Image.Transform.AFFINE,
                                         tuple(pil_inverse[:2].reshape(-1)), Image.Resampling.BILINEAR)
    im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.85, 1.15))
    im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.85, 1.15))
    im = ImageEnhance.Color(im).enhance(rng.uniform(0.9, 1.1))
    return np.asarray(im).copy(), (transformed / [width - 1, height - 1]).astype(np.float32)


def write_submission(sample, predictions, test_index, sizes, destination):
    """Fill the supplied template in its original order; never infer row ordering."""
    if set(predictions) != set(test_index):
        raise ValueError("Predicted IDs must match all test image IDs exactly")
    checked = {}
    for key in test_index:
        xy = np.asarray(predictions[key], dtype=float)
        if xy.shape != (2,) or not np.isfinite(xy).all():
            raise ValueError(f"Invalid prediction for {key}: {xy}")
        wh = np.asarray(sizes[key], dtype=float)
        if not ((xy >= 0) & (xy <= wh - 1)).all():
            raise ValueError(f"Out-of-image prediction for {key}: {xy}")
        checked[key] = xy
    result = sample.copy()
    result["value"] = [float(checked[key][axis]) for key, axis in map(parse_submission_key, sample.ImageID)]
    result.to_csv(destination, index=False, float_format="%.6f")
    loaded = pd.read_csv(destination, dtype={"ImageID": str})
    if list(loaded.columns) != list(sample.columns) or loaded.ImageID.tolist() != sample.ImageID.tolist():
        raise RuntimeError("Export changed sample columns or row order")
    if len(loaded) != len(sample) or not np.isfinite(loaded.value.to_numpy(dtype=float)).all():
        raise RuntimeError("Export contains missing/nonfinite predictions")
    return result


def save_preview(frame, points, destination, truth=None, limit=12):
    """Save an RGB contact sheet; all displayed points are original-image pixels."""
    frame = frame.iloc[:limit]
    columns, tile_w, tile_h = 4, 320, 260
    canvas = Image.new("RGB", (columns * tile_w, int(np.ceil(len(frame) / columns)) * tile_h), "#20242a")
    draw = ImageDraw.Draw(canvas)
    for j, row in enumerate(frame.itertuples()):
        with Image.open(row.path) as original:
            w, h = original.size
            image = ImageOps.contain(original.convert("RGB"), (tile_w, tile_h - 30))
        x0, y0 = (j % columns) * tile_w, (j // columns) * tile_h
        canvas.paste(image, (x0, y0))
        for xy, color in [(points[j], "#ff4040")] + ([(truth[j], "#48ff70")] if truth is not None else []):
            x, y = x0 + xy[0] * image.width / w, y0 + xy[1] * image.height / h
            draw.ellipse((x - 5, y - 5, x + 5, y + 5), outline=color, width=2)
        draw.text((x0 + 5, y0 + tile_h - 25), f"ID {row.id}  red=pred" + (" green=GT" if truth is not None else ""), fill="white")
    canvas.save(destination)


Writing /kaggle/working/fovea_data.py


In [3]:
%%writefile /kaggle/working/train_fovea.py
"""Kaggle fovea localization: ResNet18 heatmaps, grouped CV and submission.csv.

Run `python train_fovea.py --help`, or upload Fovea_Kaggle.ipynb to Kaggle.
"""
from __future__ import annotations

import argparse
import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torchvision.models import ResNet18_Weights, resnet18

from fovea_data import (assign_folds, discover_root, inspect_dataset, load_resized,
                       save_preview, transform_sample, write_submission)


@dataclass
class Config:
    data_root: str | None = None
    output_dir: str = "/kaggle/working/fovea_output"
    image_size: int = 512
    n_splits: int = 5
    folds: tuple = (0, 1, 2, 3, 4)
    epochs: int = 80
    patience: int = 20
    batch_size: int = 8
    workers: int = 2
    lr: float = 0.0003
    encoder_lr_factor: float = 0.2
    weight_decay: float = 0.0001
    sigma: float = 2.0  # Standard deviation at the 1/4-resolution heatmap.
    seed: int = 42
    pretrained: bool = True
    pretrained_path: str | None = None
    tta: bool = True
    smoke_test: bool = False
    predict_only: bool = False


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    get_worker_info().dataset.rng = np.random.default_rng(worker_seed)


class FoveaDataset(Dataset):
    def __init__(self, frame, cache, augment=False, seed=42):
        self.frame = frame.reset_index(drop=True)
        self.cache = cache
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = self.cache[row.id]
        xy = np.array([row.Fovea_X / (row.width - 1), row.Fovea_Y / (row.height - 1)], dtype=np.float32)
        if self.augment:
            image, xy = transform_sample(image, xy, self.rng)
        tensor = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))).float().div_(255)
        tensor = (tensor - tensor.new_tensor([0.485, 0.456, 0.406])[:, None, None]) / tensor.new_tensor([0.229, 0.224, 0.225])[:, None, None]
        return tensor, torch.from_numpy(xy), torch.tensor([row.width - 1, row.height - 1], dtype=torch.float32)


def loader_for(frame, cache, cfg, train=False, seed=42):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(FoveaDataset(frame, cache, train, seed), batch_size=cfg.batch_size,
                      shuffle=train, num_workers=cfg.workers, pin_memory=torch.cuda.is_available(),
                      worker_init_fn=seed_worker, generator=generator,
                      persistent_workers=cfg.workers > 0, drop_last=False)


class ConvBlock(nn.Sequential):
    def __init__(self, channels_in, channels_out):
        super().__init__(nn.Conv2d(channels_in, channels_out, 3, padding=1, bias=False),
                         nn.GroupNorm(8, channels_out), nn.ReLU(inplace=True),
                         nn.Conv2d(channels_out, channels_out, 3, padding=1, bias=False),
                         nn.GroupNorm(8, channels_out), nn.ReLU(inplace=True))


class FoveaNet(nn.Module):
    def __init__(self, encoder_weights=None, freeze_encoder_bn=False):
        super().__init__()
        self.encoder = resnet18(weights=None)
        if encoder_weights is not None:
            self.encoder.load_state_dict(encoder_weights, strict=True)
        self.encoder.fc = nn.Identity()
        self.freeze_encoder_bn = freeze_encoder_bn
        self.decode3 = ConvBlock(512 + 256, 128)
        self.decode2 = ConvBlock(128 + 128, 64)
        self.decode1 = ConvBlock(64 + 64, 64)
        self.head = nn.Conv2d(64, 1, 1)

    def train(self, mode=True):
        super().train(mode)
        if mode and self.freeze_encoder_bn:
            for module in self.encoder.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()  # Keep pretrained running statistics with tiny batches.
        return self

    def forward(self, x):
        e = self.encoder
        x = e.maxpool(e.relu(e.bn1(e.conv1(x))))
        s1 = e.layer1(x)
        s2 = e.layer2(s1)
        s3 = e.layer3(s2)
        s4 = e.layer4(s3)
        def up(tensor, skip):
            return torch.cat([F.interpolate(tensor, size=skip.shape[-2:], mode="bilinear", align_corners=False), skip], dim=1)
        x = self.decode3(up(s4, s3))
        x = self.decode2(up(x, s2))
        x = self.decode1(up(x, s1))
        return self.head(x).squeeze(1)


def heatmap_xy(logits):
    # Softmax/reductions stay in float32 under mixed precision.
    probability = logits.float().flatten(1).softmax(dim=1).view_as(logits)
    h, w = logits.shape[-2:]
    xs = torch.linspace(0, 1, w, device=logits.device)
    ys = torch.linspace(0, 1, h, device=logits.device)
    x = (probability.sum(dim=1) * xs).sum(dim=1)
    y = (probability.sum(dim=2) * ys).sum(dim=1)
    return torch.stack([x, y], dim=1)


def localization_loss(logits, xy, sigma):
    b, h, w = logits.shape
    xs = torch.arange(w, device=logits.device, dtype=torch.float32)[None, None, :]
    ys = torch.arange(h, device=logits.device, dtype=torch.float32)[None, :, None]
    dx = xs - xy[:, 0, None, None] * (w - 1)
    dy = ys - xy[:, 1, None, None] * (h - 1)
    target = torch.exp(-(dx.square() + dy.square()) / (2 * sigma**2)).flatten(1)
    target = target / target.sum(dim=1, keepdim=True).clamp_min(1e-12)
    cross_entropy = -(target * logits.float().flatten(1).log_softmax(dim=1)).sum(dim=1).mean()
    coordinate_loss = F.smooth_l1_loss(heatmap_xy(logits), xy.float(), beta=0.02)
    return cross_entropy + 5 * coordinate_loss


def get_encoder_weights(cfg):
    if not cfg.pretrained:
        print("WARNING: training from scratch; this is intended for a pipeline smoke test.", flush=True)
        return None, "random initialization"
    if cfg.pretrained_path:
        path = Path(cfg.pretrained_path)
        if not path.is_file():
            raise FileNotFoundError(path)
        weights = torch.load(path, map_location="cpu", weights_only=True)
        # Standard torchvision ResNet18 state_dict, not a checkpoint of this model.
        probe = resnet18(weights=None)
        probe.load_state_dict(weights, strict=True)
        return weights, str(path)
    try:
        weights = ResNet18_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
        return weights, "torchvision ResNet18_Weights.DEFAULT / ImageNet"
    except Exception as exc:
        raise RuntimeError("Cannot load ImageNet weights. Enable Kaggle Internet for the first run, "
                           "or attach resnet18-f37072fd.pth and set PRETRAINED_PATH/--pretrained-path. "
                           "Use --no-pretrained only if you deliberately want to train from scratch.") from exc


@torch.inference_mode()
def predict(model, loader, device, tta=True):
    model.eval()
    all_xy = []
    for image, _, _ in loader:
        image = image.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            xy = heatmap_xy(model(image))
            if tta:
                flipped = heatmap_xy(model(torch.flip(image, dims=[3])))
                flipped = torch.stack([1 - flipped[:, 0], flipped[:, 1]], dim=1)
                xy = (xy + flipped) / 2
        all_xy.append(xy.cpu().numpy())
    return np.concatenate(all_xy).astype(np.float64)


def train_fold(cfg, fold, frame, cache, weights, device, out):
    seed_everything(cfg.seed + fold)
    training, validation = frame[frame.fold != fold].copy(), frame[frame.fold == fold].copy()
    if cfg.smoke_test:
        training, validation = training.head(4), validation.head(2)
    train_loader = loader_for(training, cache, cfg, True, cfg.seed + fold)
    valid_loader = loader_for(validation, cache, cfg, False, cfg.seed + fold)
    model = FoveaNet(weights, freeze_encoder_bn=weights is not None).to(device)
    encoder_params, decoder_params = [], []
    for name, param in model.named_parameters():
        (encoder_params if name.startswith("encoder.") else decoder_params).append(param)
    optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": cfg.lr * cfg.encoder_lr_factor},
                                   {"params": decoder_params, "lr": cfg.lr}], weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    best, stalled, history = float("inf"), 0, []
    scale = validation[["width", "height"]].to_numpy(dtype=float) - 1
    truth = validation[["Fovea_X", "Fovea_Y"]].to_numpy(dtype=float)
    start = time.time()
    for epoch in range(cfg.epochs):
        model.train()
        total, count = 0.0, 0
        for image, target, _ in train_loader:
            image, target = image.to(device, non_blocking=True), target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = localization_loss(model(image), target, cfg.sigma)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Nonfinite loss in fold {fold}, epoch {epoch + 1}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            total += loss.item() * len(image)
            count += len(image)
        normalized = predict(model, valid_loader, device, cfg.tta)
        distance = np.linalg.norm(normalized * scale - truth, axis=1)
        metric = float(distance.mean())
        if not np.isfinite(metric):
            raise RuntimeError("Nonfinite validation metric")
        history.append({"fold": fold, "epoch": epoch + 1, "train_loss": total / count,
                        "mean_distance_px": metric, "median_distance_px": float(np.median(distance)),
                        "decoder_lr": optimizer.param_groups[1]["lr"]})
        print(f"fold={fold} epoch={epoch + 1:03d}/{cfg.epochs} loss={total / count:.4f} "
              f"val_mean={metric:.2f}px val_median={np.median(distance):.2f}px "
              f"elapsed={(time.time() - start) / 60:.1f}min", flush=True)
        if metric < best:
            best, stalled = metric, 0
            torch.save({"model": model.state_dict(), "fold": fold, "epoch": epoch + 1,
                        "validation_mean_px": best, "config": asdict(cfg)}, out / f"fold_{fold}_best.pth")
        else:
            stalled += 1
        scheduler.step()
        pd.DataFrame(history).to_csv(out / f"fold_{fold}_history.csv", index=False)
        if stalled >= cfg.patience:
            print(f"fold={fold}: early stopping; best={best:.2f}px", flush=True)
            break
    del model, optimizer, scaler, train_loader, valid_loader
    if device.type == "cuda":
        torch.cuda.empty_cache()


def run(cfg):
    cfg.folds = tuple(cfg.folds)
    if cfg.smoke_test:
        cfg.image_size, cfg.epochs, cfg.patience = 128, 1, 1
        cfg.folds, cfg.batch_size, cfg.workers = (0,), 2, 0
        cfg.pretrained, cfg.pretrained_path = False, None
        cfg.output_dir = str(Path(cfg.output_dir) / "smoke_test_NOT_FOR_SUBMISSION")
    if cfg.image_size < 64 or cfg.image_size % 32:
        raise ValueError("image_size must be >=64 and divisible by 32")
    if (cfg.epochs < 1 or cfg.patience < 1 or cfg.batch_size < 1 or cfg.workers < 0
            or cfg.sigma <= 0 or cfg.lr <= 0 or cfg.encoder_lr_factor <= 0):
        raise ValueError("Invalid training configuration")
    if not cfg.folds or len(set(cfg.folds)) != len(cfg.folds) or any(f < 0 or f >= cfg.n_splits for f in cfg.folds):
        raise ValueError("folds must be distinct values in range(n_splits)")
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        torch.set_num_threads(min(4, os.cpu_count() or 1))
        print("WARNING: CPU detected. Use a Kaggle GPU for a full training run.", flush=True)
    root = discover_root(cfg.data_root)
    out = Path(cfg.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    frame, sample, test_index, sizes, audit = inspect_dataset(root, out)
    frame["fold"] = assign_folds(frame, cfg.n_splits, cfg.seed)
    fold_path = out / "fold_assignments.csv"
    fold_table = frame[["id", "group", "fold"]]
    if cfg.predict_only:
        if not fold_path.is_file():
            raise FileNotFoundError("Prediction requires the original fold_assignments.csv and fold checkpoints")
        previous = pd.read_csv(fold_path, dtype={"id": str, "group": str})
        if not previous.equals(fold_table.reset_index(drop=True)):
            raise ValueError("Current dataset/seed/fold settings differ from saved fold assignments")
    else:
        fold_table.to_csv(fold_path, index=False)
    (out / ("prediction_config.json" if cfg.predict_only else "config.json")).write_text(
        json.dumps(asdict(cfg), indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(audit, indent=2, ensure_ascii=False), flush=True)
    print(f"device={device}; fold sizes={frame.fold.value_counts().sort_index().to_dict()}", flush=True)
    # Keep only compact uint8 resized images in RAM; no repeated JPEG decoding each epoch.
    cache = {row.id: load_resized(row.path, cfg.image_size) for row in frame.itertuples()}
    test = pd.DataFrame([{"id": key, "path": str(path), "width": sizes[key][0], "height": sizes[key][1],
                          "Fovea_X": 0.0, "Fovea_Y": 0.0} for key, path in test_index.items()])
    cache.update({row.id: load_resized(row.path, cfg.image_size) for row in test.itertuples()})
    weights, provenance = (None, "existing fold checkpoints") if cfg.predict_only else get_encoder_weights(cfg)
    if not cfg.predict_only:
        (out / "encoder_weights_source.txt").write_text(provenance, encoding="utf-8")
    oof = np.full((len(frame), 2), np.nan)
    baseline = np.full((len(frame), 2), np.nan)
    test_sum = np.zeros((len(test), 2), dtype=np.float64)
    summary = []
    for fold in cfg.folds:
        if not cfg.predict_only:
            train_fold(cfg, fold, frame, cache, weights, device, out)
        checkpoint_path = out / f"fold_{fold}_best.pth"
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"Missing trained checkpoint {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        saved_cfg = checkpoint["config"]
        if checkpoint["fold"] != fold or any(saved_cfg[k] != getattr(cfg, k) for k in ("image_size", "n_splits", "seed")):
            raise ValueError("Checkpoint fold/image_size/n_splits/seed differs from current settings")
        model = FoveaNet().to(device)
        model.load_state_dict(checkpoint["model"], strict=True)
        valid = frame[frame.fold == fold]
        if cfg.smoke_test:
            valid = valid.head(2)
        idx = valid.index.to_numpy()
        valid_xy = predict(model, loader_for(valid, cache, cfg), device, cfg.tta)
        scale = valid[["width", "height"]].to_numpy(dtype=float) - 1
        oof[idx] = valid_xy * scale
        train = frame[frame.fold != fold]
        train_norm = train[["Fovea_X", "Fovea_Y"]].to_numpy() / (train[["width", "height"]].to_numpy() - 1)
        baseline[idx] = np.median(train_norm, axis=0) * scale
        truth = valid[["Fovea_X", "Fovea_Y"]].to_numpy()
        errors = np.linalg.norm(oof[idx] - truth, axis=1)
        summary.append({"fold": fold, "best_epoch": checkpoint["epoch"], "n_valid": len(valid),
                        "mean_distance_px": float(errors.mean()), "median_distance_px": float(np.median(errors))})
        test_sum += predict(model, loader_for(test, cache, cfg), device, cfg.tta)
        del model, checkpoint
        if device.type == "cuda":
            torch.cuda.empty_cache()
    valid_mask = np.isfinite(oof).all(axis=1)
    truth = frame[["Fovea_X", "Fovea_Y"]].to_numpy()
    distances = np.linalg.norm(oof[valid_mask] - truth[valid_mask], axis=1)
    baseline_distances = np.linalg.norm(baseline[valid_mask] - truth[valid_mask], axis=1)
    oof_table = frame[["id", "fold", "width", "height", "Fovea_X", "Fovea_Y"]].copy()
    oof_table[["pred_x", "pred_y"]] = oof
    oof_table["distance_px"] = np.linalg.norm(oof - truth, axis=1)
    oof_table.to_csv(out / "oof_predictions.csv", index=False)
    scores = {"smoke_test": cfg.smoke_test, "n_validated": int(valid_mask.sum()), "n_train": len(frame),
              "complete_oof": bool(valid_mask.all()), "folds": summary,
              "mean_distance_px": float(distances.mean()), "median_distance_px": float(np.median(distances)),
              "p90_distance_px": float(np.percentile(distances, 90)),
              "median_coordinate_baseline_mean_px": float(baseline_distances.mean()),
              "metric_note": "Euclidean distance in ORIGINAL-image pixels; official competition metric was not supplied."}
    (out / "cv_metrics.json").write_text(json.dumps(scores, indent=2), encoding="utf-8")
    normalized_test = np.clip(test_sum / len(cfg.folds), 0.0, 1.0)
    test_xy = normalized_test * (test[["width", "height"]].to_numpy(dtype=float) - 1)
    predictions = {key: xy for key, xy in zip(test.id, test_xy)}
    name = "smoke_submission_DO_NOT_SUBMIT.csv" if cfg.smoke_test else "submission.csv"
    submission = write_submission(sample, predictions, test_index, sizes, out / name)
    test_table = test[["id", "width", "height"]].copy()
    test_table[["Fovea_X", "Fovea_Y"]] = test_xy
    test_table.to_csv(out / "test_coordinates.csv", index=False)
    save_preview(test, test_xy, out / "test_prediction_preview.jpg")
    # Show the largest validation errors first, so poor localizations are visible.
    order = np.flatnonzero(valid_mask)[np.argsort(-distances)]
    save_preview(frame.iloc[order], oof[order], out / "validation_worst_preview.jpg", truth[order])
    print(json.dumps(scores, indent=2), flush=True)
    print(submission.head(8).to_string(index=False), flush=True)
    print(f"Saved {out / name} ({len(submission)} rows).", flush=True)
    if cfg.smoke_test:
        print("SMOKE TEST ONLY: random initialization, 4 training images, 1 epoch. Do not submit this CSV.", flush=True)
    return scores


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root")
    parser.add_argument("--output-dir", default="/kaggle/working/fovea_output")
    parser.add_argument("--image-size", type=int, default=512)
    parser.add_argument("--n-splits", type=int, default=5)
    parser.add_argument("--folds", type=int, nargs="+", default=[0, 1, 2, 3, 4])
    parser.add_argument("--epochs", type=int, default=80)
    parser.add_argument("--patience", type=int, default=20)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--lr", type=float, default=0.0003)
    parser.add_argument("--pretrained-path")
    parser.add_argument("--no-pretrained", dest="pretrained", action="store_false")
    parser.add_argument("--no-tta", dest="tta", action="store_false")
    parser.add_argument("--smoke-test", action="store_true")
    parser.add_argument("--predict-only", action="store_true")
    return Config(**vars(parser.parse_args()))


if __name__ == "__main__":
    run(parse_args())


Writing /kaggle/working/train_fovea.py


In [4]:
# 支持重新运行：重新加载上面刚写入的模块。
import importlib
import fovea_data
import train_fovea
importlib.reload(fovea_data)
importlib.reload(train_fovea)
from train_fovea import Config, run

cfg = Config(data_root=DATA_ROOT, output_dir=OUTPUT_DIR, image_size=IMAGE_SIZE,
             batch_size=BATCH_SIZE, epochs=EPOCHS, patience=PATIENCE,
             n_splits=N_SPLITS, folds=tuple(FOLDS), workers=NUM_WORKERS,
             seed=SEED, pretrained=PRETRAINED, pretrained_path=PRETRAINED_PATH,
             tta=TTA, smoke_test=SMOKE_TEST, predict_only=PREDICT_ONLY)
scores = run(cfg)


RuntimeError: Found 0 datasets: []. Set DATA_ROOT explicitly.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image as DisplayImage, FileLink

out = Path(cfg.output_dir)
csv_name = "smoke_submission_DO_NOT_SUBMIT.csv" if cfg.smoke_test else "submission.csv"
submission = pd.read_csv(out / csv_name)
assert list(submission.columns) == ["ImageID", "value"]
assert submission["value"].notna().all()
assert np.isfinite(submission["value"].to_numpy()).all()
display(submission)
print(json.dumps(scores, indent=2))
if cfg.smoke_test:
    print("当前为冒烟测试：请改 SMOKE_TEST=False 后重新运行训练，才能得到正式提交。")
else:
    display(FileLink(str(out / "submission.csv")))
display(DisplayImage(filename=str(out / "test_prediction_preview.jpg")))
display(DisplayImage(filename=str(out / "validation_worst_preview.jpg")))
